# Stage 1 & 2: Data Cleaning, Preparation & Encoding

This notebook covers the cleaning, preparation, and numerical encoding of phishing and legitimate email datasets from three sources:
1. CEAS_08
2. SpamAssassin
3. Nazario

The goal is to merge them into a single, balanced dataset and prepare it for training.

In [ ]:
import pandas as pd
from sklearn.utils import resample
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import string
import os

print("Libraries imported successfully.")

## 1. Loading the datasets
We load the three CSV files, handling potential encoding issues (UTF-8 or Latin-1).

In [ ]:
def load_csv(filename):
    try:
        df = pd.read_csv(filename, encoding='utf-8', on_bad_lines='skip')
    except UnicodeDecodeError:
        df = pd.read_csv(filename, encoding='latin1', on_bad_lines='skip')
    return df

ceas = load_csv('CEAS_08.csv')
nazario = load_csv('Nazario.csv')
spam_assassin = load_csv('SpamAssasin.csv')

print(f"CEAS_08 loaded: {ceas.shape}")
print(f"Nazario loaded: {nazario.shape}")
print(f"SpamAssassin loaded: {spam_assassin.shape}")

## 2. Standardizing Columns & Labels
We identify the email text column (`body`) and the labels (`label`). Legitimate = 0, Phishing = 1.

In [ ]:
# Rename columns to standard 'email_text' and 'label'
ceas = ceas[['body', 'label']].rename(columns={'body': 'email_text'})
nazario = nazario[['body', 'label']].rename(columns={'body': 'email_text'})
spam_assassin = spam_assassin[['body', 'label']].rename(columns={'body': 'email_text'})

# Merge datasets
df_merged = pd.concat([ceas, nazario, spam_assassin], ignore_index=True)
print(f"Merged total shape: {df_merged.shape}")

## 3. Cleaning & Encoding Fixes
We remove null values, duplicates, and ensure all text is correctly encoded in UTF-8.

In [ ]:
# Remove empty values
df_merged.dropna(subset=['email_text', 'label'], inplace=True)

# Remove duplicates based on email text
df_merged.drop_duplicates(subset=['email_text'], inplace=True)

# Fix encoding (Force UTF-8)
df_merged['email_text'] = df_merged['email_text'].apply(lambda x: str(x).encode('utf-8', 'ignore').decode('utf-8'))

# Remove extremely short emails (< 10 characters)
df_merged = df_merged[df_merged['email_text'].str.strip().str.len() > 10]

print(f"Final clean dataset size: {df_merged.shape}")

## 4. Class Distribution & Balancing
We check the value counts and apply undersampling to avoid bias.

In [ ]:
print("Class distribution BEFORE balancing:")
print(df_merged['label'].value_counts())

# Balancing classes
major_class = df_merged['label'].value_counts().idxmax()
minor_class = df_merged['label'].value_counts().idxmin()

df_major = df_merged[df_merged.label == major_class]
df_minor = df_merged[df_merged.label == minor_class]

# Downsample majority
df_major_downsampled = resample(df_major, 
                               replace=False,    
                               n_samples=len(df_minor), 
                               random_state=42)

df_balanced = pd.concat([df_major_downsampled, df_minor])

print("\nClass distribution AFTER balancing:")
print(df_balanced['label'].value_counts())

## 5. Metadata Pre-processing & TF-IDF Labeling (Encoding)
To prepare for models, we convert text to numbers using **TF-IDF Vectorization** (Term Frequency-Inverse Document Frequency).

In [ ]:
def clean_for_ml(text):
    text = text.lower()
    text = re.sub(f'[{re.escape(string.punctuation)}]', '', text)
    text = re.sub(r'\d+', '', text)
    return text

# Apply pre-processing
df_balanced['email_text_clean'] = df_balanced['email_text'].apply(clean_for_ml)

# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X = tfidf.fit_transform(df_balanced['email_text_clean'])
y = df_balanced['label']

print(f"Encoded features shape: {X.shape}")

## 6. Saving Clean Data

In [ ]:
df_balanced.to_csv('emails.csv', index=False)
print("Saved cleaned and balanced data to emails.csv")